In [0]:
%pip install openai unidecode gspread==5.12.4 tiktoken mlflow
%restart_python
%load_ext autoreload
%autoreload 2 

In [0]:
from abc import ABC, abstractmethod
import time
import mlflow
from contextlib import contextmanager
import json
import pandas as pd
import datetime
import re
import openai
from openai import OpenAI
import gspread
import random
import logging
import os
from unidecode import unidecode
from pathlib import Path
import pyspark
from pyspark.sql.functions import *
from pyspark.sql.types import *
from functools import reduce
from typing import *
import tiktoken
import io
import sys


# Add the parent directory of 'localizers' to sys.path
project_root = Path('/Workspace/Users/krista@jamcity.com/CentralizedLocalizationWorkflow/localizers')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

### Localizer Modules
from general_config import *
from ml_tracker import MLTracker
from publishing_config import *
from publishing_localizer import *

In [0]:
%run "./authenticationScript"

In [0]:
def setup_widgets():
    dbutils.widgets.text('RowFingerprint',"")
    dbutils.widgets.text("Timestamp","")
    dbutils.widgets.text("SubmitterEmail","")
    dbutils.widgets.text("DueDate","")
    dbutils.widgets.text('LocType',"")
    dbutils.widgets.text("Game","")
    dbutils.widgets.text("TargetLanguages","")
    dbutils.widgets.text("URL","")
    dbutils.widgets.text("QAFlag","")
    dbutils.widgets.text("Status","")
    dbutils.widgets.text("LastStatusUpdate","")

def get_request():
    return {'Timestamp':dbutils.widgets.get('Timestamp'), 
       'SubmitterEmail':dbutils.widgets.get('SubmitterEmail'),
       'DueDate': dbutils.widgets.get("DueDate"), 
       'Game':  dbutils.widgets.get("Game"),
       'TargetLanguages':  dbutils.widgets.get("TargetLanguages"), 
       'URL': dbutils.widgets.get("URL"), 
       'LocType': dbutils.widgets.get("LocType"), 
       'RowFingerprint': dbutils.widgets.get("RowFingerprint")}

setup_widgets()

In [0]:
#TODO update authenticatinon script and secrets!!
gsheet_client = get_gspread_client_from_secret_old()
gpt_client = get_model_client() 

In [0]:
#rqst = get_request()

cfg = {
    "input": 
        {
            "required_tabs": ["ios","android"],
            "ios_header_rows": 3, 
            "android_header_rows": 3
        },
    ##add more formatting data for header rows
    "char_limit_policy": "strict",
    "output_sheets":
        ["formatted ios", "formatted android", "long results",'wide results'],
    ##add more formatting info for output sheets
    "qc": {"enabled": True, "max_retries": 5,}}

In [0]:
rqsts = [
    {
        "Timestamp": "10/20/2025",
        "SubmitterEmail": "vanessa.mendoza@jamcity.com",
        "DueDate": "11/10/2025",
        "Game": "Harry Potter: Hogwarts Mystery",
        "TargetLanguages": "Spanish (Latin America), French (France), German, Korean, Italian, Japanese, Simplified Chinese, Traditional Chinese (Taiwan), Portuguese (Brazil)",
        "URL": "https://docs.google.com/spreadsheets/d/1qXVY3nsC3ar-EDKROdWxCfXRHrBLUcfiByTPAnnrCno/edit?usp=sharing",
        "QA Flag": "Yes",
        "LocType": "Publishing",
        "Status": "SUBMITTED",
        "LastStatusUpdate": "2025-10-20T17:39:10.035Z",
        "RowFingerprint": "PdP/7zQhGhDMpuf7afuTsg=="
    },
    {
        "Timestamp": "10/20/2025",
        "SubmitterEmail": "vanessa.mendoza@jamcity.com",
        "DueDate": "11/20/2025",
        "Game": "Disney Emoji Blitz",
        "TargetLanguages": "Spanish (Latin America), French (France), German, Russian, Korean, Italian, Japanese, Simplified Chinese, Traditional Chinese (Taiwan), Portuguese (Brazil)",
        "URL": "https://docs.google.com/spreadsheets/d/1SOeTCmFnbvgeYtx6M3wYgpSaVca4C0oTzs8TPJ59XOM/edit?usp=sharing",
        "QA Flag": "Yes",
        "LocType": "Publishing",
        "Status": "SUBMITTED",
        "LastStatusUpdate": "2025-10-20T19:39:05.150Z",
        "RowFingerprint": "ayGX0Rf+FNLMxg1w7uO0Wg=="
    },
    {
        "Timestamp": "10/20/2025",
        "SubmitterEmail": "vanessa.mendoza@jamcity.com",
        "DueDate": "11/10/2025",
        "Game": "Disney Magic Match",
        "TargetLanguages": "Spanish (Latin America), French (France), German, Russian, Korean, Italian, Japanese, Simplified Chinese, Traditional Chinese (Taiwan), Portuguese (Brazil)",
        "URL": "https://docs.google.com/spreadsheets/d/1UJOuhJie5cuI7O9n3X8na9ALe-Gn8c3Jxv_QId7nK0w/edit?usp=sharing",
        "QA Flag": "Yes",
        "LocType": "Publishing",
        "Status": "SUBMITTED",
        "LastStatusUpdate": "2025-10-20T19:39:05.570Z",
        "RowFingerprint": "qXUbAwB0Ud3CPNB5KfbXDQ=="
    },
    {
        "Timestamp": "10/20/2025",
        "SubmitterEmail": "vanessa.mendoza@jamcity.com",
        "DueDate": "11/10/2025",
        "Game": "Cookie Jam",
        "TargetLanguages": "Spanish (Latin America), French (France), German, Russian, Korean, Italian, Japanese, Simplified Chinese, Traditional Chinese (Taiwan), Portuguese (Brazil)",
        "URL": "https://docs.google.com/spreadsheets/d/1WIsiWOMIZdy8lmdNgfKMt8qzWPkGReeA1igAs5FosMA/edit?usp=sharing",
        "QA Flag": "Yes",
        "LocType": "Publishing",
        "Status": "SUBMITTED",
        "LastStatusUpdate": "2025-10-20T19:39:06.296Z",
        "RowFingerprint": "fKkxXqW6ce03vkTu6tjQWA=="
    },
    {
        "Timestamp": "10/20/2025",
        "SubmitterEmail": "vanessa.mendoza@jamcity.com",
        "DueDate": "11/10/2025",
        "Game": "Panda Pop",
        "TargetLanguages": "Spanish (Latin America), French (France), German, Russian, Korean, Italian, Japanese, Simplified Chinese, Traditional Chinese (Taiwan), Portuguese (Brazil)",
        "URL": "https://docs.google.com/spreadsheets/d/1Yxj4yb0u9qVPmvExlQajr6leMh4A_43gHtcb-Rj4q0Q/edit?usp=sharing",
        "QA Flag": "Yes",
        "LocType": "Publishing",
        "Status": "SUBMITTED",
        "LastStatusUpdate": "2025-10-20T19:39:06.632Z",
        "RowFingerprint": "PUzha8JJPYE69Hkd/c8iAw=="
    }
]

In [0]:
localizers =[PublishingLocalizer(request = rqst, 
                                 gsheet_client =gsheet_client, gpt_client = gpt_client, cfg=cfg) for rqst in rqsts] 

In [0]:
#localizer = PublishingLocalizer(request = rqst, gsheet_client =gsheet_client, gpt_client = gpt_client, cfg=cfg)

In [0]:
# index 0,1,2,3 done

localizer = localizers[4]

In [0]:
localizer.validate_inputs()
localizer.load_inputs()
preprocessed = localizer.preprocess()
groups = localizer.build_prompts()
raw_results = localizer.translate(groups)

In [0]:
""" 

# Special case when only android exists without ios - needs to be updated in the main function 

processed = localizer.postprocess(raw_results)

wide,long = localizer._helper_parse_row_idx(processed)

### Special case
again_processed = []
for df in processed:

    cols = df.columns
    for col in cols:
        if col not in ['row_idx','row_id','game','platform','type_desc','en_char_limit','language_cd']:
            df = df.rename(columns={col: "translation"})
    again_processed.append(df)


long_results = pd.concat(again_processed)

unioned_results = long_results.merge(localizer.android_long_df.toPandas(), on = ['row_id','game','platform','en_char_limit'], how='left')"""

In [0]:

processed = localizer.postprocess(raw_results)


wide,long = localizer._helper_parse_row_idx(processed)

unioned_inputs = localizer._helper_long_inputs_for_merge()

long_results = pd.concat(long)


unioned_results = localizer.unioned_inputs.merge(long_results, on = ['row_id','game','platform','en_char_limit'],how='left')

In [0]:
import numpy as np
import pandas as pd


rev_lang_map = dict(zip(localizer.lang_cds, localizer.languages))


unioned_results['language'] = unioned_results['language_cd'].map(rev_lang_map)



# Example logic
unioned_results["target_char_limit"] = np.where(
    unioned_results["language_cd"].isin(["ja_JP", "ko_KR", "zh_CN", "zh_TW"]) & (unioned_results["platform"] == "android"),
    unioned_results["en_char_limit"] / 2,
    unioned_results["en_char_limit"]
)



long_results_order = ["RowFingerprint","row_idx",'row_id','en_char_limit','game','platform','type_desc','en_US','language','language_cd','target_char_limit','translation']
unioned_results = unioned_results[long_results_order]

In [0]:
unioned_results.display()

In [0]:
values = unioned_results[long_results_order].values.tolist()

val_range = f"A2:L{len(values)+1}"

wksht = localizer.sh.worksheet('long results')

wksht.batch_update([{'range':val_range, 'values':values}])